<a href="https://colab.research.google.com/github/TAlkam/NASS_2017/blob/main/NASS_2017_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NASS 2017 Alzheimer’s Ambulatory Surgery Study
# Exploratory Data Analysis for Prepared Age/Sex-Matched Dataset
# ============================================================

!pip install pyreadstat openpyxl -q

In [ ]:
# ============================================================
# 1. Import libraries
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyreadstat
from google.colab import files

In [ ]:
# ============================================================
# 2. Upload the prepared matched Stata dataset
# ============================================================

uploaded = files.upload()

# Automatically detect uploaded .dta file
dta_files = [f for f in uploaded.keys() if f.endswith(".dta")]

if len(dta_files) == 0:
    raise FileNotFoundError("No .dta file uploaded. Please upload your matched NASS 2017 Stata file.")

file_path = dta_files[0]

print("Uploaded file:", file_path)

In [ ]:
# ============================================================
# 3. Load Stata dataset
# ============================================================

df, meta = pyreadstat.read_dta(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

In [ ]:
# ============================================================
# 4. Show variable names and labels
# ============================================================

var_labels = pd.DataFrame({
    "Variable": meta.column_names,
    "Label": [meta.column_labels[i] if i < len(meta.column_labels) else "" for i in range(len(meta.column_names))]
})

display(var_labels)

var_labels.to_excel("Variable_labels.xlsx", index=False)

In [ ]:
# ============================================================
# 5. Basic dataset structure
# ============================================================

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.reset_index().rename(columns={"index": "Variable", 0: "Data type"}))

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# ============================================================
# 6. Standardize key variable names to lowercase
# ============================================================

df.columns = df.columns.str.lower()

print("Variable names converted to lowercase.")
print(df.columns.tolist())

In [ ]:
# ============================================================
# 7. Confirm key variables exist
# ============================================================

key_vars = [
    "alz",
    "age",
    "female",
    "dispuniform",
    "totchg",
    "i10_ndx",
    "ncpt_inscope",
    "pay1",
    "zipinc_qrtl",
    "pl_nchs",
    "aweekend",
    "dqtr"
]

available_key_vars = [v for v in key_vars if v in df.columns]
missing_key_vars = [v for v in key_vars if v not in df.columns]

print("Available key variables:")
print(available_key_vars)

print("\nMissing key variables:")
print(missing_key_vars)

In [ ]:
# ============================================================
# 8. Confirm Alzheimer’s group balance
# ============================================================

if "alz" not in df.columns:
    raise ValueError("Variable 'alz' not found. Please confirm that your prepared dataset includes the Alzheimer’s indicator.")

df["alz"] = df["alz"].astype(int)

df["alz_label"] = df["alz"].map({
    0: "Non-Alzheimer's",
    1: "Alzheimer's disease"
})

alz_counts = df["alz_label"].value_counts().reset_index()
alz_counts.columns = ["Group", "N"]

alz_counts["Percent"] = 100 * alz_counts["N"] / alz_counts["N"].sum()

display(alz_counts)

In [ ]:
# ============================================================
# 9. Confirm sex matching
# ============================================================

if "female" in df.columns:
    sex_table_n = pd.crosstab(df["female"], df["alz_label"])
    sex_table_pct = pd.crosstab(df["female"], df["alz_label"], normalize="columns") * 100

    print("Sex counts:")
    display(sex_table_n)

    print("Sex column percentages:")
    display(sex_table_pct.round(2))
else:
    print("Variable 'female' not found.")

In [ ]:
# ============================================================
# 10. Confirm age matching
# ============================================================

if "age" in df.columns:
    age_summary = df.groupby("alz_label")["age"].agg(
        N="count",
        Mean="mean",
        SD="std",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Min="min",
        Max="max"
    ).reset_index()

    display(age_summary)
else:
    print("Variable 'age' not found.")

In [ ]:
# ============================================================
# 11. Missingness summary
# ============================================================

missing_summary = pd.DataFrame({
    "Variable": df.columns,
    "Missing_N": df.isna().sum().values,
    "Missing_Percent": (df.isna().sum().values / len(df)) * 100
})

missing_summary = missing_summary.sort_values("Missing_Percent", ascending=False)

display(missing_summary.head(40))

missing_summary.to_excel("Missingness_summary.xlsx", index=False)

In [ ]:
# ============================================================
# 12. Create primary outcome: non-routine discharge
# ============================================================

# Common DISPUNIFORM coding:
# 1  = Routine discharge
# 2  = Transfer to short-term hospital
# 5  = Other transfer, including skilled nursing/intermediate care/other facility
# 6  = Home health care
# 7  = Against medical advice
# 20 = Died
# 99 = Alive, destination unknown

if "dispuniform" not in df.columns:
    raise ValueError("Variable 'dispuniform' not found. Please check the disposition variable name.")

df["nonroutine"] = np.nan

df.loc[df["dispuniform"] == 1, "nonroutine"] = 0
df.loc[df["dispuniform"].isin([2, 5, 6, 7, 20, 99]), "nonroutine"] = 1

df["nonroutine_label"] = df["nonroutine"].map({
    0: "Routine discharge",
    1: "Non-routine discharge"
})

print("Non-routine discharge distribution:")
display(df["nonroutine_label"].value_counts(dropna=False).reset_index().rename(
    columns={"index": "Disposition", "nonroutine_label": "N"}
))

In [ ]:
# ============================================================
# 13. Non-routine discharge by Alzheimer’s status
# ============================================================

discharge_counts = pd.crosstab(df["alz_label"], df["nonroutine_label"], margins=True)
discharge_row_pct = pd.crosstab(df["alz_label"], df["nonroutine_label"], normalize="index") * 100

print("Discharge counts:")
display(discharge_counts)

print("Discharge row percentages:")
display(discharge_row_pct.round(2))

discharge_counts.to_excel("EDA_discharge_counts_by_AD_status.xlsx")
discharge_row_pct.to_excel("EDA_discharge_percent_by_AD_status.xlsx")

In [ ]:
# ============================================================
# 14. Total charges summary
# ============================================================

if "totchg" in df.columns:
    df["log_totchg"] = np.where(df["totchg"] > 0, np.log(df["totchg"]), np.nan)

    charge_summary = df.groupby("alz_label")["totchg"].agg(
        N="count",
        Mean="mean",
        SD="std",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Min="min",
        Max="max"
    ).reset_index()

    display(charge_summary)

    charge_summary.to_excel("EDA_total_charges_by_AD_status.xlsx", index=False)
else:
    print("Variable 'totchg' not found.")

In [ ]:
# ============================================================
# 15. High-charge encounter variable
# ============================================================

if "totchg" in df.columns:
    charge_cutoff_90 = df["totchg"].quantile(0.90)
    df["high_charge"] = np.where(df["totchg"] >= charge_cutoff_90, 1, 0)

    print("90th percentile total charge cutoff:", charge_cutoff_90)

    high_charge_counts = pd.crosstab(df["alz_label"], df["high_charge"], margins=True)
    high_charge_pct = pd.crosstab(df["alz_label"], df["high_charge"], normalize="index") * 100

    print("High-charge counts:")
    display(high_charge_counts)

    print("High-charge row percentages:")
    display(high_charge_pct.round(2))

    high_charge_counts.to_excel("EDA_high_charge_counts_by_AD_status.xlsx")
    high_charge_pct.to_excel("EDA_high_charge_percent_by_AD_status.xlsx")
else:
    print("Variable 'totchg' not found.")

In [ ]:
# ============================================================
# 16. Diagnosis burden and procedure burden
# ============================================================

burden_vars = [v for v in ["i10_ndx", "ncpt_inscope"] if v in df.columns]

burden_summary_list = []

for var in burden_vars:
    temp = df.groupby("alz_label")[var].agg(
        N="count",
        Mean="mean",
        SD="std",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Min="min",
        Max="max"
    ).reset_index()

    temp.insert(0, "Variable", var)
    burden_summary_list.append(temp)

if len(burden_summary_list) > 0:
    burden_summary = pd.concat(burden_summary_list, axis=0)
    display(burden_summary)

    burden_summary.to_excel("EDA_diagnosis_procedure_burden_by_AD_status.xlsx", index=False)
else:
    print("No burden variables found.")

In [ ]:
# ============================================================
# 17. Categorical variables by Alzheimer’s status
# ============================================================

categorical_vars = [
    "female",
    "pay1",
    "zipinc_qrtl",
    "pl_nchs",
    "aweekend",
    "dqtr",
    "dispuniform",
    "nonroutine",
    "high_charge"
]

categorical_vars = [v for v in categorical_vars if v in df.columns]

cat_tables = {}

for var in categorical_vars:
    counts = pd.crosstab(df[var], df["alz_label"], margins=True)
    pct = pd.crosstab(df[var], df["alz_label"], normalize="columns") * 100

    cat_tables[var] = {
        "counts": counts,
        "percent": pct
    }

    print(f"\nVariable: {var}")
    print("Counts:")
    display(counts)

    print("Column percentages:")
    display(pct.round(2))

In [ ]:
# ============================================================
# 18. Export categorical EDA tables
# ============================================================

with pd.ExcelWriter("EDA_categorical_tables_by_AD_status.xlsx") as writer:
    for var, tables in cat_tables.items():
        tables["counts"].to_excel(writer, sheet_name=f"{var}_counts"[:31])
        tables["percent"].round(2).to_excel(writer, sheet_name=f"{var}_pct"[:31])

print("Categorical EDA tables exported.")

In [ ]:
# ============================================================
# 19. Procedure category analysis using cptccs1
# ============================================================

if "cptccs1" in df.columns:
    proc_counts = pd.crosstab(df["cptccs1"], df["alz_label"])
    proc_counts["Total"] = proc_counts.sum(axis=1)
    proc_counts = proc_counts.sort_values("Total", ascending=False)

    display(proc_counts.head(30))

    proc_counts.to_excel("EDA_top_procedure_categories_cptccs1.xlsx")

    # Alzheimer’s-only top procedure categories
    top_proc_ad = (
        df[df["alz"] == 1]["cptccs1"]
        .value_counts()
        .head(20)
        .reset_index()
    )

    top_proc_ad.columns = ["cptccs1", "N"]

    display(top_proc_ad)

    top_proc_ad.to_excel("EDA_top_procedure_categories_AD_only.xlsx", index=False)
else:
    print("Variable 'cptccs1' not found.")

In [ ]:
# ============================================================
# 20. CPT code analysis using cpt1
# ============================================================

if "cpt1" in df.columns:
    cpt1_counts = pd.crosstab(df["cpt1"], df["alz_label"])
    cpt1_counts["Total"] = cpt1_counts.sum(axis=1)
    cpt1_counts = cpt1_counts.sort_values("Total", ascending=False)

    display(cpt1_counts.head(30))

    cpt1_counts.to_excel("EDA_top_CPT1_codes_by_AD_status.xlsx")

    top_cpt1_ad = (
        df[df["alz"] == 1]["cpt1"]
        .value_counts()
        .head(20)
        .reset_index()
    )

    top_cpt1_ad.columns = ["cpt1", "N"]

    display(top_cpt1_ad)

    top_cpt1_ad.to_excel("EDA_top_CPT1_codes_AD_only.xlsx", index=False)
else:
    print("Variable 'cpt1' not found.")

In [ ]:
# ============================================================
# 21. ICD-10 diagnosis code frequency in Alzheimer’s group
# ============================================================

dx_cols = [c for c in df.columns if c.startswith("i10_dx")]

print("Diagnosis columns found:", dx_cols)

if len(dx_cols) > 0:
    ad_dx_values = []

    ad_df = df[df["alz"] == 1].copy()

    for col in dx_cols:
        values = ad_df[col].dropna().astype(str).str.strip().str.upper()
        values = values[values != ""]
        ad_dx_values.extend(values.tolist())

    ad_dx_counts = pd.Series(ad_dx_values).value_counts().reset_index()
    ad_dx_counts.columns = ["ICD10_Code", "N"]

    display(ad_dx_counts.head(50))

    ad_dx_counts.to_excel("EDA_top_ICD10_codes_AD_group.xlsx", index=False)
else:
    print("No ICD-10 diagnosis variables found.")

In [ ]:
# ============================================================
# 22. Exclude G30 codes to see non-Alzheimer’s comorbid diagnoses in AD group
# ============================================================

if len(dx_cols) > 0:
    ad_comorbidity_counts = ad_dx_counts[
        ~ad_dx_counts["ICD10_Code"].str.startswith("G30")
    ].copy()

    display(ad_comorbidity_counts.head(50))

    ad_comorbidity_counts.to_excel("EDA_top_non_G30_diagnoses_AD_group.xlsx", index=False)

In [ ]:
# ============================================================
# 23. Basic visualizations
# ============================================================

# Age distribution by group
if "age" in df.columns:
    plt.figure(figsize=(8, 5))
    df[df["alz"] == 0]["age"].dropna().plot(kind="hist", alpha=0.5, bins=20, label="Non-Alzheimer's")
    df[df["alz"] == 1]["age"].dropna().plot(kind="hist", alpha=0.5, bins=20, label="Alzheimer's disease")
    plt.xlabel("Age")
    plt.ylabel("Number of encounters")
    plt.title("Age Distribution by Alzheimer’s Status")
    plt.legend()
    plt.tight_layout()
    plt.savefig("Figure_age_distribution_by_AD_status.png", dpi=300)
    plt.show()

In [ ]:
# Total charges boxplot by group
if "totchg" in df.columns:
    plot_df = df[["alz_label", "totchg"]].dropna().copy()

    plt.figure(figsize=(7, 5))
    plot_df.boxplot(column="totchg", by="alz_label")
    plt.xlabel("Group")
    plt.ylabel("Total charges")
    plt.title("Total Charges by Alzheimer’s Status")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig("Figure_total_charges_boxplot_by_AD_status.png", dpi=300)
    plt.show()

In [ ]:
# Log charges boxplot by group
if "log_totchg" in df.columns:
    plot_df = df[["alz_label", "log_totchg"]].dropna().copy()

    plt.figure(figsize=(7, 5))
    plot_df.boxplot(column="log_totchg", by="alz_label")
    plt.xlabel("Group")
    plt.ylabel("Log total charges")
    plt.title("Log Total Charges by Alzheimer’s Status")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig("Figure_log_total_charges_boxplot_by_AD_status.png", dpi=300)
    plt.show()

In [ ]:
# Non-routine discharge bar plot
if "nonroutine_label" in df.columns:
    discharge_pct = (
        pd.crosstab(df["alz_label"], df["nonroutine_label"], normalize="index") * 100
    )

    discharge_pct.plot(kind="bar", figsize=(8, 5))
    plt.xlabel("Group")
    plt.ylabel("Percent")
    plt.title("Discharge Disposition by Alzheimer’s Status")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig("Figure_discharge_by_AD_status.png", dpi=300)
    plt.show()

In [ ]:
# Diagnosis burden boxplot
if "i10_ndx" in df.columns:
    plot_df = df[["alz_label", "i10_ndx"]].dropna().copy()

    plt.figure(figsize=(7, 5))
    plot_df.boxplot(column="i10_ndx", by="alz_label")
    plt.xlabel("Group")
    plt.ylabel("Number of ICD-10 diagnoses")
    plt.title("Diagnosis Burden by Alzheimer’s Status")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig("Figure_diagnosis_burden_by_AD_status.png", dpi=300)
    plt.show()

In [ ]:
# Procedure burden boxplot
if "ncpt_inscope" in df.columns:
    plot_df = df[["alz_label", "ncpt_inscope"]].dropna().copy()

    plt.figure(figsize=(7, 5))
    plot_df.boxplot(column="ncpt_inscope", by="alz_label")
    plt.xlabel("Group")
    plt.ylabel("Number of in-scope CPT procedures")
    plt.title("Procedure Burden by Alzheimer’s Status")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig("Figure_procedure_burden_by_AD_status.png", dpi=300)
    plt.show()

In [ ]:
# Top procedure categories in Alzheimer’s group
if "cptccs1" in df.columns:
    top_proc = df[df["alz"] == 1]["cptccs1"].value_counts().head(15)

    plt.figure(figsize=(10, 6))
    top_proc.sort_values().plot(kind="barh")
    plt.xlabel("Number of encounters")
    plt.ylabel("CPT CCS procedure category")
    plt.title("Top Procedure Categories Among Alzheimer’s Encounters")
    plt.tight_layout()
    plt.savefig("Figure_top_procedure_categories_AD.png", dpi=300)
    plt.show()

In [ ]:
# ============================================================
# 24. Create a combined EDA summary workbook
# ============================================================

with pd.ExcelWriter("NASS_2017_AD_matched_EDA_summary.xlsx") as writer:
    alz_counts.to_excel(writer, sheet_name="AD_group_counts", index=False)

    if "age" in df.columns:
        age_summary.to_excel(writer, sheet_name="Age_summary", index=False)

    if "totchg" in df.columns:
        charge_summary.to_excel(writer, sheet_name="Charges_summary", index=False)

    if len(burden_summary_list) > 0:
        burden_summary.to_excel(writer, sheet_name="Burden_summary", index=False)

    discharge_counts.to_excel(writer, sheet_name="Discharge_counts")
    discharge_row_pct.round(2).to_excel(writer, sheet_name="Discharge_percent")

    if "high_charge" in df.columns:
        high_charge_counts.to_excel(writer, sheet_name="High_charge_counts")
        high_charge_pct.round(2).to_excel(writer, sheet_name="High_charge_percent")

    if "cptccs1" in df.columns:
        proc_counts.head(50).to_excel(writer, sheet_name="Top_CPTCCS1")

    if "cpt1" in df.columns:
        cpt1_counts.head(50).to_excel(writer, sheet_name="Top_CPT1")

    if len(dx_cols) > 0:
        ad_dx_counts.head(100).to_excel(writer, sheet_name="Top_ICD10_AD", index=False)
        ad_comorbidity_counts.head(100).to_excel(writer, sheet_name="Top_nonG30_AD", index=False)

    missing_summary.to_excel(writer, sheet_name="Missingness", index=False)
    var_labels.to_excel(writer, sheet_name="Variable_labels", index=False)

print("EDA workbook saved: NASS_2017_AD_matched_EDA_summary.xlsx")

In [ ]:
# ============================================================
# 25. Download EDA outputs
# ============================================================

import zipfile

output_files = [
    "Variable_labels.xlsx",
    "Missingness_summary.xlsx",
    "EDA_discharge_counts_by_AD_status.xlsx",
    "EDA_discharge_percent_by_AD_status.xlsx",
    "EDA_total_charges_by_AD_status.xlsx",
    "EDA_high_charge_counts_by_AD_status.xlsx",
    "EDA_high_charge_percent_by_AD_status.xlsx",
    "EDA_diagnosis_procedure_burden_by_AD_status.xlsx",
    "EDA_categorical_tables_by_AD_status.xlsx",
    "EDA_top_procedure_categories_cptccs1.xlsx",
    "EDA_top_procedure_categories_AD_only.xlsx",
    "EDA_top_CPT1_codes_by_AD_status.xlsx",
    "EDA_top_CPT1_codes_AD_only.xlsx",
    "EDA_top_ICD10_codes_AD_group.xlsx",
    "EDA_top_non_G30_diagnoses_AD_group.xlsx",
    "NASS_2017_AD_matched_EDA_summary.xlsx",
    "Figure_age_distribution_by_AD_status.png",
    "Figure_total_charges_boxplot_by_AD_status.png",
    "Figure_log_total_charges_boxplot_by_AD_status.png",
    "Figure_discharge_by_AD_status.png",
    "Figure_diagnosis_burden_by_AD_status.png",
    "Figure_procedure_burden_by_AD_status.png",
    "Figure_top_procedure_categories_AD.png"
]

zip_name = "NASS_2017_AD_matched_EDA_outputs.zip"

with zipfile.ZipFile(zip_name, "w") as zipf:
    for f in output_files:
        if os.path.exists(f):
            zipf.write(f)

files.download(zip_name)

alz
alz_label
nonroutine
nonroutine_label
log_totchg
high_charge

In [ ]:
# ============================================================
# 1. Install and import packages
# ============================================================

!pip install xgboost shap openpyxl -q

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
import shap

In [ ]:
# ============================================================
# 2. Confirm / recreate primary and secondary outcomes
# ============================================================

# Primary outcome: non-routine discharge
# 0 = routine discharge
# 1 = transfer, home health, AMA, death, unknown/other destination

if "nonroutine" not in df.columns:
    df["nonroutine"] = np.nan
    df.loc[df["dispuniform"] == 1, "nonroutine"] = 0
    df.loc[df["dispuniform"].isin([2, 5, 6, 7, 20, 99]), "nonroutine"] = 1

df["nonroutine_label"] = df["nonroutine"].map({
    0: "Routine discharge",
    1: "Non-routine discharge"
})

# Log total charges
if "log_totchg" not in df.columns:
    df["log_totchg"] = np.where(df["totchg"] > 0, np.log(df["totchg"]), np.nan)

# High-charge encounter: top 10% of total charges
if "high_charge" not in df.columns:
    charge_cutoff_90 = df["totchg"].quantile(0.90)
    df["high_charge"] = np.where(df["totchg"] >= charge_cutoff_90, 1, 0)
else:
    charge_cutoff_90 = df["totchg"].quantile(0.90)

print("90th percentile charge cutoff:", charge_cutoff_90)

print("\nPrimary outcome:")
display(pd.crosstab(df["alz_label"], df["nonroutine_label"], margins=True))

print("\nPrimary outcome row percentages:")
display((pd.crosstab(df["alz_label"], df["nonroutine_label"], normalize="index") * 100).round(2))

print("\nHigh-charge row percentages:")
display((pd.crosstab(df["alz_label"], df["high_charge"], normalize="index") * 100).round(2))

In [ ]:
# ============================================================
# 3. Crude odds ratio: Alzheimer’s vs non-Alzheimer’s
# ============================================================

tab = pd.crosstab(df["alz"], df["nonroutine"])

# Ensure table has columns 0 and 1
a = tab.loc[1, 1]  # AD + nonroutine
b = tab.loc[1, 0]  # AD + routine
c = tab.loc[0, 1]  # non-AD + nonroutine
d = tab.loc[0, 0]  # non-AD + routine

or_crude = (a / b) / (c / d)
se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_low = np.exp(np.log(or_crude) - 1.96 * se_log_or)
ci_high = np.exp(np.log(or_crude) + 1.96 * se_log_or)

crude_or_table = pd.DataFrame({
    "Comparison": ["Alzheimer’s vs non-Alzheimer’s"],
    "OR": [or_crude],
    "95% CI lower": [ci_low],
    "95% CI upper": [ci_high]
})

display(crude_or_table)

crude_or_table.to_excel("Crude_OR_nonroutine_discharge.xlsx", index=False)

In [ ]:
# ============================================================
# 4. Prepare regression dataset
# ============================================================

# Candidate covariates based on your NASS variables
candidate_covariates = [
    "alz",
    "age",
    "female",
    "pay1",
    "zipinc_qrtl",
    "pl_nchs",
    "i10_ndx",
    "ncpt_inscope",
    "aweekend",
    "dqtr",
    "cptccs1"
]

available_covariates = [v for v in candidate_covariates if v in df.columns]

print("Available covariates:")
print(available_covariates)

# Regression dataset for primary outcome
reg_vars = ["nonroutine"] + available_covariates
reg_df = df[reg_vars].dropna().copy()

# Convert categorical variables
categorical_covariates = [
    "pay1",
    "zipinc_qrtl",
    "pl_nchs",
    "aweekend",
    "dqtr",
    "cptccs1"
]

for v in categorical_covariates:
    if v in reg_df.columns:
        reg_df[v] = reg_df[v].astype("category")

print("Regression dataset shape:", reg_df.shape)
print(reg_df["nonroutine"].value_counts())

In [ ]:
# ============================================================
# 5. Logistic regression models for non-routine discharge
# ============================================================

# Model 1: Alzheimer’s only
formula1 = "nonroutine ~ alz"

# Model 2: Alzheimer’s + age + sex
formula2 = "nonroutine ~ alz + age + female"

# Model 3: add payer, income, location
formula3 = formula2
for v in ["pay1", "zipinc_qrtl", "pl_nchs"]:
    if v in reg_df.columns:
        formula3 += f" + C({v})"

# Model 4: add diagnosis and procedure burden
formula4 = formula3
for v in ["i10_ndx", "ncpt_inscope"]:
    if v in reg_df.columns:
        formula4 += f" + {v}"

# Model 5: add timing variables
formula5 = formula4
for v in ["aweekend", "dqtr"]:
    if v in reg_df.columns:
        formula5 += f" + C({v})"

# Model 6: add procedure category
formula6 = formula5
if "cptccs1" in reg_df.columns:
    formula6 += " + C(cptccs1)"

formulas = {
    "Model 1: Alzheimer’s only": formula1,
    "Model 2: Age + sex": formula2,
    "Model 3: Socioeconomic/location": formula3,
    "Model 4: Burden adjusted": formula4,
    "Model 5: Timing adjusted": formula5,
    "Model 6: Procedure adjusted": formula6
}

models = {}

for name, formula in formulas.items():
    print("\nRunning:", name)
    print(formula)
    try:
        model = smf.logit(formula, data=reg_df).fit(disp=False, maxiter=200)
        models[name] = model
        print("Converged.")
    except Exception as e:
        print("Model failed:", e)

In [ ]:
# ============================================================
# 6. Export odds ratios from logistic regression
# ============================================================

def logistic_or_table(model, model_name):
    params = model.params
    conf = model.conf_int()
    pvals = model.pvalues

    out = pd.DataFrame({
        "Model": model_name,
        "Variable": params.index,
        "OR": np.exp(params.values),
        "CI_lower": np.exp(conf[0].values),
        "CI_upper": np.exp(conf[1].values),
        "p_value": pvals.values
    })

    return out

or_tables = []

for name, model in models.items():
    or_tables.append(logistic_or_table(model, name))

or_results = pd.concat(or_tables, axis=0)

# Show Alzheimer’s result across models first
alz_or_results = or_results[or_results["Variable"] == "alz"].copy()

display(alz_or_results)
display(or_results)

or_results.to_excel("Table3_logistic_regression_nonroutine_discharge_all_ORs.xlsx", index=False)
alz_or_results.to_excel("Table3_Alzheimer_OR_across_models.xlsx", index=False)

In [ ]:
# ============================================================
# 7. Linear regression for log total charges
# ============================================================

charge_vars = ["log_totchg"] + available_covariates
charge_df = df[charge_vars].dropna().copy()

for v in categorical_covariates:
    if v in charge_df.columns:
        charge_df[v] = charge_df[v].astype("category")

charge_formula = "log_totchg ~ alz + age + female"

for v in ["pay1", "zipinc_qrtl", "pl_nchs"]:
    if v in charge_df.columns:
        charge_formula += f" + C({v})"

for v in ["i10_ndx", "ncpt_inscope"]:
    if v in charge_df.columns:
        charge_formula += f" + {v}"

for v in ["aweekend", "dqtr"]:
    if v in charge_df.columns:
        charge_formula += f" + C({v})"

if "cptccs1" in charge_df.columns:
    charge_formula += " + C(cptccs1)"

print(charge_formula)

charge_model = smf.ols(charge_formula, data=charge_df).fit(cov_type="HC3")

print(charge_model.summary())

charge_results = pd.DataFrame({
    "Variable": charge_model.params.index,
    "Beta": charge_model.params.values,
    "CI_lower": charge_model.conf_int()[0].values,
    "CI_upper": charge_model.conf_int()[1].values,
    "p_value": charge_model.pvalues.values
})

# Approximate percent difference for log outcome
charge_results["Approx_percent_difference"] = (np.exp(charge_results["Beta"]) - 1) * 100

display(charge_results)

charge_results.to_excel("Table4_linear_regression_log_total_charges.xlsx", index=False)

In [ ]:
# ============================================================
# Fix sparse CPTCCS1 categories before high-charge regression
# ============================================================

high_charge_df = df[high_charge_vars].dropna().copy()

# Collapse rare cptccs1 categories
if "cptccs1" in high_charge_df.columns:
    counts = high_charge_df["cptccs1"].value_counts()
    common_cats = counts[counts >= 30].index   # you can change 30 to 50 if needed

    high_charge_df["cptccs1_collapsed"] = high_charge_df["cptccs1"].where(
        high_charge_df["cptccs1"].isin(common_cats),
        "Other"
    )

# Convert categorical variables
for v in ["pay1", "zipinc_qrtl", "pl_nchs", "aweekend", "dqtr", "cptccs1_collapsed"]:
    if v in high_charge_df.columns:
        high_charge_df[v] = high_charge_df[v].astype("category")

# Rebuild formula using collapsed procedure category
high_charge_formula = "high_charge ~ alz + age + female"

for v in ["pay1", "zipinc_qrtl", "pl_nchs"]:
    if v in high_charge_df.columns:
        high_charge_formula += f" + C({v})"

for v in ["i10_ndx", "ncpt_inscope"]:
    if v in high_charge_df.columns:
        high_charge_formula += f" + {v}"

for v in ["aweekend", "dqtr"]:
    if v in high_charge_df.columns:
        high_charge_formula += f" + C({v})"

if "cptccs1_collapsed" in high_charge_df.columns:
    high_charge_formula += " + C(cptccs1_collapsed)"

print(high_charge_formula)

high_charge_model = smf.logit(
    high_charge_formula,
    data=high_charge_df
).fit(method="bfgs", disp=False, maxiter=500)

high_charge_or = logistic_or_table(
    high_charge_model,
    "High-charge encounter model with collapsed procedure categories"
)

display(high_charge_or)

high_charge_or.to_excel(
    "Table5_logistic_regression_high_charge_collapsed_procedure.xlsx",
    index=False
)

In [ ]:
# ============================================================
# 9. Top procedure categories by Alzheimer’s status
# ============================================================

if "cptccs1" in df.columns:
    proc_counts = pd.crosstab(df["cptccs1"], df["alz_label"])
    proc_counts["Total"] = proc_counts.sum(axis=1)
    proc_counts = proc_counts.sort_values("Total", ascending=False)

    display(proc_counts.head(30))

    proc_counts.head(50).to_excel("Table2_top_procedure_categories_by_AD_status.xlsx")

    # Row percentages among each Alzheimer’s group
    proc_pct = pd.crosstab(df["cptccs1"], df["alz_label"], normalize="columns") * 100
    proc_pct = proc_pct.loc[proc_counts.index]

    display(proc_pct.head(30).round(2))

    proc_pct.head(50).round(2).to_excel("Table2_top_procedure_categories_percent_by_AD_status.xlsx")

In [ ]:
# ============================================================
# 10. Machine learning dataset for non-routine discharge
# ============================================================

ml_features = []

candidate_ml_features = [
    "alz",
    "age",
    "female",
    "pay1",
    "zipinc_qrtl",
    "pl_nchs",
    "i10_ndx",
    "ncpt_inscope",
    "aweekend",
    "dqtr",
    "cptccs1"
]

for v in candidate_ml_features:
    if v in df.columns:
        ml_features.append(v)

ml_df = df[ml_features + ["nonroutine"]].dropna().copy()

X = ml_df[ml_features]
y = ml_df["nonroutine"].astype(int)

categorical_features = [
    v for v in ml_features
    if v in ["pay1", "zipinc_qrtl", "pl_nchs", "aweekend", "dqtr", "cptccs1"]
]

numeric_features = [
    v for v in ml_features
    if v not in categorical_features
]

print("ML dataset shape:", ml_df.shape)
print("Outcome distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
# ============================================================
# 11. Train/test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=20260524,
    stratify=y
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

print("\nTrain outcome distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest outcome distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
# ============================================================
# 12. Preprocessing and ML models
# ============================================================

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

ml_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="lbfgs"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=20260524,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=20260524,
        n_jobs=-1
    )
}

In [ ]:
# ============================================================
# 13. Train and evaluate ML models
# ============================================================

ml_results = []
trained_pipelines = {}

for name, model in ml_models.items():
    print("\n========================================")
    print("Training:", name)
    print("========================================")

    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    result = {
        "Model": name,
        "AUROC": roc_auc_score(y_test, y_prob),
        "AUPRC": average_precision_score(y_test, y_prob),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    }

    ml_results.append(result)
    trained_pipelines[name] = pipe

    print(result)
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

ml_results_df = pd.DataFrame(ml_results).sort_values("AUROC", ascending=False)

display(ml_results_df)

ml_results_df.to_excel("Table6_ML_model_performance_nonroutine_discharge.xlsx", index=False)

In [ ]:
# ============================================================
# 14. ROC and precision-recall curves for best model
# ============================================================

best_model_name = ml_results_df.iloc[0]["Model"]
best_pipe = trained_pipelines[best_model_name]

print("Best model by AUROC:", best_model_name)

y_prob_best = best_pipe.predict_proba(X_test)[:, 1]

plt.figure(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_test, y_prob_best)
plt.title(f"ROC Curve: {best_model_name}")
plt.tight_layout()
plt.savefig("Figure_ROC_best_model_nonroutine.png", dpi=300)
plt.show()

plt.figure(figsize=(6, 6))
PrecisionRecallDisplay.from_predictions(y_test, y_prob_best)
plt.title(f"Precision-Recall Curve: {best_model_name}")
plt.tight_layout()
plt.savefig("Figure_PR_best_model_nonroutine.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 15. Confusion matrix for best model
# ============================================================

y_pred_best = best_pipe.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

cm_df = pd.DataFrame(
    cm,
    index=["Actual routine", "Actual non-routine"],
    columns=["Predicted routine", "Predicted non-routine"]
)

display(cm_df)

cm_df.to_excel("Table7_confusion_matrix_best_model.xlsx")

In [ ]:
# ============================================================
# 16. SHAP analysis for XGBoost
# ============================================================

xgb_pipe = trained_pipelines["XGBoost"]

# Transform train/test data
X_train_transformed = xgb_pipe.named_steps["preprocess"].transform(X_train)
X_test_transformed = xgb_pipe.named_steps["preprocess"].transform(X_test)

# Get feature names
feature_names = xgb_pipe.named_steps["preprocess"].get_feature_names_out()

# Convert sparse matrix to dense if needed
if hasattr(X_test_transformed, "toarray"):
    X_test_dense = X_test_transformed.toarray()
else:
    X_test_dense = X_test_transformed

xgb_model = xgb_pipe.named_steps["model"]

# SHAP explainer
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_dense)

print("SHAP values shape:", np.array(shap_values).shape)
print("X_test shape:", X_test_dense.shape)

In [ ]:
# ============================================================
# 17. SHAP summary dot plot
# ============================================================

plt.figure()
shap.summary_plot(
    shap_values,
    X_test_dense,
    feature_names=feature_names,
    show=False,
    max_display=20
)

plt.tight_layout()
plt.savefig("Figure_SHAP_summary_dotplot_nonroutine.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 18. SHAP bar plot
# ============================================================

plt.figure()
shap.summary_plot(
    shap_values,
    X_test_dense,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20
)

plt.tight_layout()
plt.savefig("Figure_SHAP_barplot_nonroutine.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 19. Export SHAP feature importance table
# ============================================================

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_abs_SHAP": mean_abs_shap
}).sort_values("Mean_abs_SHAP", ascending=False)

display(shap_importance.head(30))

shap_importance.to_excel("Table8_SHAP_feature_importance_nonroutine.xlsx", index=False)

In [ ]:
# ============================================================
# 20. Alzheimer’s-only analysis: routine vs non-routine discharge
# ============================================================

ad_df = df[df["alz"] == 1].copy()

print("Alzheimer’s-only sample size:", ad_df.shape)

# Continuous variables by discharge outcome
ad_cont_vars = [v for v in ["age", "totchg", "log_totchg", "i10_ndx", "ncpt_inscope"] if v in ad_df.columns]

ad_cont_summary = []

for var in ad_cont_vars:
    temp = ad_df.groupby("nonroutine_label")[var].agg(
        N="count",
        Mean="mean",
        SD="std",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Min="min",
        Max="max"
    ).reset_index()

    temp.insert(0, "Variable", var)
    ad_cont_summary.append(temp)

ad_cont_summary = pd.concat(ad_cont_summary, axis=0)

display(ad_cont_summary)

ad_cont_summary.to_excel("Table9_AD_only_continuous_by_discharge_outcome.xlsx", index=False)

# Categorical variables by discharge outcome
ad_cat_vars = [v for v in ["female", "pay1", "zipinc_qrtl", "pl_nchs", "aweekend", "dqtr", "cptccs1"] if v in ad_df.columns]

with pd.ExcelWriter("Table10_AD_only_categorical_by_discharge_outcome.xlsx") as writer:
    for var in ad_cat_vars:
        counts = pd.crosstab(ad_df[var], ad_df["nonroutine_label"])
        pct = pd.crosstab(ad_df[var], ad_df["nonroutine_label"], normalize="columns") * 100

        counts.to_excel(writer, sheet_name=f"{var}_counts"[:31])
        pct.round(2).to_excel(writer, sheet_name=f"{var}_pct"[:31])

In [ ]:
# ============================================================
# 21. Manuscript-ready summary of key findings
# ============================================================

summary_rows = []

# Non-routine discharge
nonroutine_tab = pd.crosstab(df["alz_label"], df["nonroutine"], normalize="index") * 100

summary_rows.append({
    "Finding": "Non-routine discharge (%)",
    "Alzheimer’s disease": nonroutine_tab.loc["Alzheimer's disease", 1],
    "Non-Alzheimer’s": nonroutine_tab.loc["Non-Alzheimer's", 1]
})

# Total charges
charge_summary = df.groupby("alz_label")["totchg"].median()

summary_rows.append({
    "Finding": "Median total charges",
    "Alzheimer’s disease": charge_summary.loc["Alzheimer's disease"],
    "Non-Alzheimer’s": charge_summary.loc["Non-Alzheimer's"]
})

# Diagnosis burden
dx_summary = df.groupby("alz_label")["i10_ndx"].median()

summary_rows.append({
    "Finding": "Median number of ICD-10 diagnoses",
    "Alzheimer’s disease": dx_summary.loc["Alzheimer's disease"],
    "Non-Alzheimer’s": dx_summary.loc["Non-Alzheimer's"]
})

# Procedure burden
proc_summary = df.groupby("alz_label")["ncpt_inscope"].median()

summary_rows.append({
    "Finding": "Median number of in-scope CPT procedures",
    "Alzheimer’s disease": proc_summary.loc["Alzheimer's disease"],
    "Non-Alzheimer’s": proc_summary.loc["Non-Alzheimer's"]
})

key_findings_table = pd.DataFrame(summary_rows)

display(key_findings_table)

key_findings_table.to_excel("Table11_key_findings_summary.xlsx", index=False)

In [ ]:
import pyreadstat
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# Create readable SHAP feature names
# Run this BEFORE the SHAP plot
# ============================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Manual readable labels for the cptccs1 categories shown in your SHAP plot
# Please verify any category labels against your HCUP CCS procedure documentation if needed.
cptccs1_label_map = {
    3: "Laminectomy / disc excision",
    6: "Peripheral nerve decompression",
    9: "Other nervous system procedures",
    10: "Thyroidectomy",
    12: "Other endocrine procedures",
    15: "Lens and cataract procedures",
    16: "Retinal tear/detachment repair",
    80: "Appendectomy",
    86: "Other hernia repair",
    112: "Other urinary tract procedures",
    132: "Other female genital procedures",
    142: "Partial excision of bone",
    143: "Bunionectomy / toe deformity repair",
    147: "Lower-extremity fracture/dislocation treatment",
    148: "Other fracture/dislocation procedure",
    151: "Knee meniscal cartilage excision",
    152: "Knee arthroplasty",
    158: "Spinal fusion",
    167: "Mastectomy",
    175: "Other skin/breast procedures"
}

other_feature_label_map = {
    "alz": "Alzheimer’s disease",
    "age": "Age",
    "female": "Female sex",
    "i10_ndx": "Diagnosis count",
    "ncpt_inscope": "Procedure count",
    "pay1": "Primary payer",
    "zipinc_qrtl": "ZIP income quartile",
    "pl_nchs": "Patient residence location",
    "aweekend": "Weekend procedure",
    "dqtr": "Calendar quarter"
}

def readable_feature_name(feature_name):
    f = str(feature_name)

    # Remove sklearn prefixes
    f = f.replace("cat__", "").replace("num__", "")

    # Convert cptccs1 one-hot labels
    if f.startswith("cptccs1_"):
        code_raw = f.replace("cptccs1_", "")
        match = re.search(r"\d+", code_raw)

        if match:
            code = int(match.group())
            return cptccs1_label_map.get(code, f"CCS procedure category {code}")

        return f

    # Convert other variables
    for base_var, readable_base in other_feature_label_map.items():
        if f == base_var:
            return readable_base

        if f.startswith(base_var + "_"):
            category = f.replace(base_var + "_", "")
            return f"{readable_base}: {category}"

    return f

# Create pretty feature names
pretty_feature_names = [readable_feature_name(f) for f in feature_names]

# Check result
feature_name_mapping = pd.DataFrame({
    "Original_feature": feature_names,
    "Readable_feature": pretty_feature_names
})

display(feature_name_mapping.head(50))

# Save mapping
feature_name_mapping.to_excel("SHAP_feature_name_mapping_readable.xlsx", index=False)

print("pretty_feature_names created successfully.")
print("Number of feature names:", len(pretty_feature_names))
print("Number of SHAP columns:", np.array(shap_values).shape[-1])

In [ ]:
# ============================================================
# SHAP summary dot plot with readable labels
# ============================================================

plt.figure(figsize=(9, 8))

shap.summary_plot(
    shap_values,
    X_test_dense,
    feature_names=pretty_feature_names,
    show=False,
    max_display=20
)

plt.title("Explainable Machine Learning Predictors of Non-routine Discharge", fontsize=13)
plt.tight_layout()
plt.savefig("Figure_SHAP_summary_dotplot_readable_medical_labels.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# SHAP bar plot with human-readable medical labels
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Calculate mean absolute SHAP values
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Create readable SHAP importance table
shap_bar_df = pd.DataFrame({
    "Feature": pretty_feature_names,
    "Mean_abs_SHAP": mean_abs_shap
})

# Sort by importance and keep top 20
shap_bar_df = shap_bar_df.sort_values("Mean_abs_SHAP", ascending=False).head(20)

# Reverse order for horizontal bar plot
shap_bar_df_plot = shap_bar_df.iloc[::-1]

# Plot
plt.figure(figsize=(10, 8))
plt.barh(
    shap_bar_df_plot["Feature"],
    shap_bar_df_plot["Mean_abs_SHAP"]
)

plt.xlabel("Mean absolute SHAP value")
plt.ylabel("")
plt.title("Top Predictors of Non-routine Discharge After Ambulatory Surgery")
plt.tight_layout()

plt.savefig("Figure_SHAP_barplot_readable_medical_labels.png", dpi=300, bbox_inches="tight")
plt.show()

# Export table
shap_bar_df.to_excel("Table_SHAP_barplot_readable_medical_labels.xlsx", index=False)

display(shap_bar_df)

In [ ]:
# ============================================================
# Figure 1. Discharge disposition by Alzheimer's status
# Fixed legend, percentage labels, and guaranteed plot output
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Recreate alz_label if needed
if "alz_label" not in df.columns:
    df["alz_label"] = df["alz"].map({
        0: "Non-Alzheimer's",
        1: "Alzheimer's disease"
    })

# Recreate nonroutine if needed
if "nonroutine" not in df.columns:
    df["nonroutine"] = np.nan
    df.loc[df["dispuniform"] == 1, "nonroutine"] = 0
    df.loc[df["dispuniform"].isin([2, 5, 6, 7, 20, 99]), "nonroutine"] = 1

# Recreate nonroutine_label
df["nonroutine_label"] = df["nonroutine"].map({
    0: "Routine discharge",
    1: "Non-routine discharge"
})

# Remove missing outcome rows for plotting
plot_df = df.dropna(subset=["alz_label", "nonroutine_label"]).copy()

# Create percent table
discharge_pct = (
    pd.crosstab(plot_df["alz_label"], plot_df["nonroutine_label"], normalize="index") * 100
)

# Set row and column order
row_order = ["Alzheimer's disease", "Non-Alzheimer's"]
col_order = ["Non-routine discharge", "Routine discharge"]

discharge_pct = discharge_pct.reindex(index=row_order)
discharge_pct = discharge_pct[[c for c in col_order if c in discharge_pct.columns]]

print(discharge_pct.round(2))

# Plot
ax = discharge_pct.plot(kind="bar", figsize=(9, 5), width=0.75)

ax.set_xlabel("Group")
ax.set_ylabel("Percent")
ax.set_title("Discharge Disposition by Alzheimer’s Status")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Put legend outside plot area
ax.legend(
    title="Discharge disposition",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3, fontsize=9)

plt.tight_layout()
plt.savefig("Figure1_discharge_disposition_fixed.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Complete Table 1 safely: Baseline characteristics by AD status
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Create clean Alzheimer’s labels
# ------------------------------------------------------------

df["alz"] = df["alz"].astype(int)

df["alz_label_clean"] = df["alz"].map({
    1: "Alzheimer's disease",
    0: "Non-Alzheimer's"
})

group_order = ["Alzheimer's disease", "Non-Alzheimer's"]

# ------------------------------------------------------------
# 2. Optional readable category labels
# ------------------------------------------------------------

pay1_labels = {
    1: "Medicare",
    2: "Medicaid",
    3: "Private insurance",
    4: "Self-pay",
    5: "No charge",
    6: "Other"
}

zipinc_labels = {
    1: "Quartile 1, lowest income",
    2: "Quartile 2",
    3: "Quartile 3",
    4: "Quartile 4, highest income"
}

aweekend_labels = {
    0: "Weekday procedure",
    1: "Weekend procedure"
}

dqtr_labels = {
    1: "January–March",
    2: "April–June",
    3: "July–September",
    4: "October–December"
}

# If you are not sure about pl_nchs labels, keep numeric categories
# or relabel them later after checking the HCUP coding manual.
label_maps = {
    "pay1": pay1_labels,
    "zipinc_qrtl": zipinc_labels,
    "aweekend": aweekend_labels,
    "dqtr": dqtr_labels
}

# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def format_n_pct(n, pct):
    return f"{int(n):,} ({pct:.2f}%)"

def categorical_summary(data, var, label_name):
    temp = data.copy()

    # Apply readable category labels if available
    if var in label_maps:
        temp[var + "_label"] = temp[var].map(label_maps[var]).fillna(temp[var].astype(str))
        var_use = var + "_label"
    else:
        temp[var + "_label"] = temp[var].astype(str)
        var_use = var + "_label"

    tab_n = pd.crosstab(temp[var_use], temp["alz_label_clean"])
    tab_pct = pd.crosstab(temp[var_use], temp["alz_label_clean"], normalize="columns") * 100

    # Make sure both columns exist
    for g in group_order:
        if g not in tab_n.columns:
            tab_n[g] = 0
            tab_pct[g] = 0

    tab_n = tab_n[group_order]
    tab_pct = tab_pct[group_order]

    rows = []

    for category in tab_n.index:
        rows.append({
            "Characteristic": f"{label_name}: {category}",
            "Alzheimer's disease N=6,968": format_n_pct(tab_n.loc[category, "Alzheimer's disease"], tab_pct.loc[category, "Alzheimer's disease"]),
            "Non-Alzheimer's N=6,968": format_n_pct(tab_n.loc[category, "Non-Alzheimer's"], tab_pct.loc[category, "Non-Alzheimer's"])
        })

    return pd.DataFrame(rows)

def median_iqr_summary(data, var, label_name):
    rows = {"Characteristic": label_name}

    for g in group_order:
        x = data.loc[data["alz_label_clean"] == g, var].dropna()
        median = x.median()
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        rows[f"{g} N=6,968"] = f"{median:.0f} ({q1:.0f}–{q3:.0f})"

    return pd.DataFrame([rows])

# ------------------------------------------------------------
# 4. Basic matched rows
# ------------------------------------------------------------

basic_rows = pd.DataFrame({
    "Characteristic": [
        "Age, mean ± SD",
        "Female, n (%)",
        "Male/sex=0, n (%)"
    ],
    "Alzheimer's disease N=6,968": [
        "79.17 ± 7.28",
        "3,727 (53.49%)",
        "3,241 (46.51%)"
    ],
    "Non-Alzheimer's N=6,968": [
        "79.17 ± 7.28",
        "3,727 (53.49%)",
        "3,241 (46.51%)"
    ]
})

# ------------------------------------------------------------
# 5. Categorical rows
# ------------------------------------------------------------

table_parts = [basic_rows]

if "pay1" in df.columns:
    table_parts.append(categorical_summary(df, "pay1", "Primary payer"))

if "zipinc_qrtl" in df.columns:
    table_parts.append(categorical_summary(df, "zipinc_qrtl", "ZIP income quartile"))

if "pl_nchs" in df.columns:
    table_parts.append(categorical_summary(df, "pl_nchs", "Patient location"))

if "aweekend" in df.columns:
    table_parts.append(categorical_summary(df, "aweekend", "Surgery timing"))

if "dqtr" in df.columns:
    table_parts.append(categorical_summary(df, "dqtr", "Quarter of year"))

# ------------------------------------------------------------
# 6. Continuous burden rows
# ------------------------------------------------------------

if "i10_ndx" in df.columns:
    table_parts.append(
        median_iqr_summary(df, "i10_ndx", "ICD-10 diagnosis count, median (IQR)")
    )

if "ncpt_inscope" in df.columns:
    table_parts.append(
        median_iqr_summary(df, "ncpt_inscope", "In-scope CPT procedure count, median (IQR)")
    )

# ------------------------------------------------------------
# 7. Final Table 1
# ------------------------------------------------------------

table1_final = pd.concat(table_parts, axis=0).reset_index(drop=True)

display(table1_final)

table1_final.to_excel("Table1_baseline_characteristics_completed.xlsx", index=False)